In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/rot-dataset/twitter_dataset.csv
/kaggle/input/rot-dataset/clean_twitter_dataset.csv
/kaggle/input/twitter-vaccination-dataset/master.csv
/kaggle/input/twitter-vaccination-dataset/vaccination2.csv


In [2]:
raw_data = pd.read_csv("/kaggle/input/rot-dataset/twitter_dataset.csv")
df = raw_data.copy()

/tmp/ipykernel_21/3674116857.py:1: DtypeWarning: Columns (0,1,2,3,4,5,6,8,9,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv("/kaggle/input/rot-dataset/twitter_dataset.csv")


In [3]:
print("N. utenti", df.username.nunique())
print("N. tweet", df.tweet.nunique())

N. utenti 41413
N. tweet 39680


In [4]:
import nltk 
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
!rm -rf /usr/share/nltk_data/corpora/wordnet
!unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Archive:  /usr/share/nltk_data/corpora/wordnet.zip
   creating: /usr/share/nltk_data/corpora/wordnet/
  inflating: /usr/share/nltk_data/corpora/wordnet/lexnames  
  inflating: /usr/share/nltk_data/corpora/wordnet/data.verb  
  inflating: /usr/share/nltk_data/corpora/wordnet/index.adv  
  inflating: /usr/share/nltk_data/corpora/wordnet/adv.exc  
  inflating: /usr/share/nltk_data/corpora/wordnet/index.verb  
  inflating: /usr/share/nltk_data/corpora/wordnet/cntlist.rev  
  inflating: /usr/share/nltk_data/corpora/wordnet/data.adj  
  inflating: /usr/share/nltk_data/corpora/wordnet/index.adj  
  inflating: /usr/share/nltk_data/corpora/w

In [5]:
import string

import nltk
from nltk import re, WordNetLemmatizer
from nltk.corpus import stopwords

MIN_YEAR = 1900
MAX_YEAR = 2100


def get_url_pattern():
    return re.compile(
        r'(https?:\/\/(?:www\.|(?!www))[a-zA-Z0-9][a-zA-Z0-9-]+[a-zA-Z0-9]\.[^\s]{2,}|https?:\/\/(?:www\.|(?!www))'
        r'[a-zA-Z0-9]\.[^\s]{2,}|www\.[a-zA-Z0-9]\.[^\s]{2,})')


def get_emojis_pattern():
    try:
        # UCS-4
        emojis_pattern = re.compile(u'([\U00002600-\U000027BF])|([\U0001f300-\U0001f64F])|([\U0001f680-\U0001f6FF])')
    except re.error:
        # UCS-2
        emojis_pattern = re.compile(
            u'([\u2600-\u27BF])|([\uD83C][\uDF00-\uDFFF])|([\uD83D][\uDC00-\uDE4F])|([\uD83D][\uDE80-\uDEFF])')
    return emojis_pattern


def get_hashtags_pattern():
    return re.compile(r'#\w+')


def get_single_letter_words_pattern():
    return re.compile(r'(?<![\w\-])\w(?![\w\-])')


def get_blank_spaces_pattern():
    return re.compile(r'\s{2,}|\t')


def get_twitter_reserved_words_pattern():
    return re.compile(r'(RT|rt|FAV|fav|VIA|via)')


def get_mentions_pattern():
    return re.compile(r'@\w*')


def is_year(text):
    if (len(text) == 3 or len(text) == 4) and (MIN_YEAR < len(text) < MAX_YEAR):
        return True
    else:
        return False


class TwitterPreprocessor:

    def __init__(self, text: str):
        self.text = text

    def fully_preprocess(self):
        return self \
            .remove_urls() \
            .remove_mentions() \
            .remove_twitter_reserved_words() \
            .remove_punctuation() \
            .remove_single_letter_words() \
            .remove_blank_spaces() \
            .stem() \
            .remove_stopwords() \
            .remove_numbers()

    def remove_urls(self):
        self.text = re.sub(pattern=get_url_pattern(), repl='', string=self.text)
        return self

    def remove_punctuation(self):
        self.text = self.text.translate(str.maketrans('', '', string.punctuation))
        return self

    def remove_mentions(self):
        self.text = re.sub(pattern=get_mentions_pattern(), repl='', string=self.text)
        return self

    def remove_hashtags(self):
        self.text = re.sub(pattern=get_hashtags_pattern(), repl='', string=self.text)
        return self

    def remove_twitter_reserved_words(self):
        self.text = re.sub(pattern=get_twitter_reserved_words_pattern(), repl='', string=self.text)
        return self

    def remove_single_letter_words(self):
        self.text = re.sub(pattern=get_single_letter_words_pattern(), repl='', string=self.text)
        return self

    def remove_blank_spaces(self):
        self.text = re.sub(pattern=get_blank_spaces_pattern(), repl=' ', string=self.text)
        return self

    def remove_stopwords(self, extra_stopwords=None):
        if extra_stopwords is None:
            extra_stopwords = []
        text = nltk.word_tokenize(self.text)
        stop_words = set(stopwords.words('english'))

        new_sentence = []
        for w in text:
            if w not in stop_words and w not in extra_stopwords:
                new_sentence.append(w)
        self.text = ' '.join(new_sentence)
        return self

    def remove_numbers(self, preserve_years=False):
        text_list = self.text.split(' ')
        for text in text_list:
            if text.isnumeric():
                if preserve_years:
                    if not is_year(text):
                        text_list.remove(text)
                else:
                    text_list.remove(text)

        self.text = ' '.join(text_list)
        return self

    def stem(self):
        lemmatizer = WordNetLemmatizer()
        text = nltk.word_tokenize(self.text)
        new_sentence = []
        for w in text:
            new_sentence.append(lemmatizer.lemmatize(w))

        self.text = ' '.join(new_sentence)
        return self

    def lowercase(self):
        self.text = self.text.lower()
        return self


In [6]:
import nltk
import pandas as pd


def preprocess(df):
    """
    Preprocessing operations on Twitter vaccination dataset. Uses the class
    TwitterPreprocessor found at <a href="https://www.kaggle.com/datasets/keplaxo/twitter-vaccination-dataset"></a>.
    Result is stored inside column clean_tweet of preprocessed_tweets.csv.
    """
    df["clean_tweet"] = df.apply(
        lambda row:
        TwitterPreprocessor(row["tweet"])
        .fully_preprocess().text,
        axis=1)

    return df

def filter_en_tweets(df:pd.DataFrame):
    return df[df.language == 'en']

In [7]:
df = preprocess(df)
df.head()

,id,conversation_id,created_at,date,time,timezone,user_id,username,name,place,...,video,near,geo,source,user_rt_id,user_rt,retweet_id,reply_to,retweet_date,clean_tweet
0,1280655184729866242,NaN,NaN,NaN,NaN,NaN,NaN,Carlosbaseball7,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,['charliekirk11'],NaN,NaN,NaN,Anyone like Ilhan Omar call literal DISMANTLIN...
1,1280655192120340481,NaN,NaN,NaN,NaN,NaN,NaN,CedricKnight12,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,['charliekirk11'],NaN,NaN,NaN,Anyone like Ilhan Omar call literal DISMANTLIN...
2,1280655206192201730,NaN,NaN,NaN,NaN,NaN,NaN,Rosaabel71,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,['charliekirk11'],NaN,NaN,NaN,Anyone like Ilhan Omar call literal DISMANTLIN...
3,1280655208729784320,NaN,NaN,NaN,NaN,NaN,NaN,LBICommUSA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,['charliekirk11'],NaN,NaN,NaN,Anyone like Ilhan Omar call literal DISMANTLIN...
4,1280655209568616448,NaN,NaN,NaN,NaN,NaN,NaN,buffalobill4729,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,['charliekirk11'],NaN,NaN,NaN,Anyone like Ilhan Omar call literal DISMANTLIN...


In [8]:
df.to_pickle("preprocessed.pth")